# GRPO Training Setup

This section prepares the environment for **Grouped Reinforcement Preference Optimization (GRPO)** training on our unified summarization dataset (`summary_corpus_merged.json`).  
We will use the **Unsloth** framework together with **vLLM** for efficient fine-tuning and inference.




In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [2]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    get_vllm, get_triton = ("vllm==0.10.1", "triton==3.2.0") if is_t4 else ("vllm", "triton")
    !uv pip install -qqq --upgrade \
        unsloth {get_vllm} {get_numpy} torchvision bitsandbytes xformers
    !uv pip install -qqq {get_triton}
!uv pip install transformers==4.55.4
!uv pip install --no-deps trl==0.22.2

> **Colab Notice:**  
> Occasionally, Colab may prompt you to restart the runtime during installation with pip .  
> Please wait until all installation commands have fully completed (all `pip install` lines finish successfully) before restarting. Restarting too early may interrupt dependency setup and cause import errors later.

In [ ]:
!pip install "datasets" "evaluate" "rouge_score" "nltk" "tqdm"

!pip install "unsloth[vllm] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo
!pip install --upgrade --force-reinstall wandb
!pip install --upgrade vllm

### Environment and Configuration Setup

In this section, we import all required libraries, initialize Weights & Biases (W&B) for experiment tracking, and define global training parameters.

For training purposes, we chose a maximum sequence length of 4096 tokens.
To avoid artificially constraining GRPO rollouts and reward evaluation, we allocate a large completion window (800 tokens).
This ensures that the model has sufficient freedom to generate longer summaries or exploratory outputs during optimization,
without truncation bias or early cutoffs that could distort GRPO’s reward estimation.

The remaining prompt window (3296 tokens) provides ample context space for source documents while keeping GPU memory usage stable.

Make sure your W&B API key is correctly set inside `wandb.login(key='')` before running this cell.


In [ ]:
import sys
import torch
import transformers
import unsloth
import vllm
import wandb
import os
import json
import re
import numpy as np
from collections import defaultdict
from typing import List

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from datasets import load_from_disk, Dataset, DatasetDict
from nltk.tokenize import sent_tokenize, word_tokenize
from rouge_score import rouge_scorer
from tqdm import tqdm
from transformers import set_seed
import random
import wandb

from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from vllm import SamplingParams
import wandb
import torch
import evaluate

print("\n=== Environment Versions ===")
print(f"Python:        {sys.version.split()[0]}")
print(f"Torch:         {torch.__version__}")
print(f"Transformers:  {transformers.__version__}")
print(f"Unsloth:       {unsloth.__version__}")
print(f"vLLM:          {vllm.__version__}")
print(f"W&B:           {wandb.__version__}")
print("=============================\n")

wandb.login(key='')
WANDB_PROJECT     = "grpo-summarization"

BASE_MODEL_NAME   = "meta-llama/Llama-3.2-1B-Instruct"
MAX_SEQ_LENGTH    = 4096
MAX_COMPLETION    = 800
MAX_PROMPT_LENGTH = MAX_SEQ_LENGTH - MAX_COMPLETION  # 3296

LORA_RANK         = 32

OUTPUT_DIR        = "outputs"
LORA_SAVE_DIR     = "grpo_saved_lora"
DATASET_DISK_DIR  = "Dataset/processed_dataset_grpo"
RUN_NAME_SUFFIX   = "llama3.2-1b-grpo"


SEED = 3407

set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


=== Environment Versions ===
Python:        3.12.12
Torch:         2.8.0+cu126
Transformers:  4.55.4
Unsloth:       2025.10.8
vLLM:          0.11.0
W&B:           0.22.2



/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mattia_maranzana (mattia_maranzana-University of bologna) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


### Model Loading and LoRA Configuration

We now load the base model and apply LoRA (Low-Rank Adaptation) for parameter-efficient fine-tuning.

- **Base model:** `meta-llama/Llama-3.2-1B-Instruct`  
- **Quantization:** loaded in **4-bit precision** to reduce memory footprint while maintaining good performance.  
- **Fast inference:** enabled through *Unsloth’s* optimized model loading.  
- **LoRA rank (`r`):** set to `32`, allowing adaptation of attention and feedforward layers with minimal added parameters.  
- **Target modules:** standard transformer projection layers (`q_proj`, `k_proj`, `v_proj`, etc.) are adapted.  
- **Gradient checkpointing:** activated to handle long-context sequences efficiently during GRPO fine-tuning.

We decided to use LoRA because full fine-tuning of large models would be too memory-intensive for GRPO,  
which already performs multiple rollouts per batch. LoRA provides a good trade-off between performance and efficiency.

After applying LoRA, the model is ready for the GRPO training phase.


In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name          = BASE_MODEL_NAME,
    max_seq_length      = MAX_SEQ_LENGTH,
    load_in_4bit        = True,
    fast_inference      = True,
    max_lora_rank       = LORA_RANK,
    gpu_memory_utilization = 0.6,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",
    ],
    lora_alpha=LORA_RANK * 2,
    use_gradient_checkpointing="unsloth",  # long-context finetuning
    random_state=3407,
)
print("Model ready with LoRA.")

INFO 10-22 15:30:59 [vllm_utils.py:694] Unsloth: Patching vLLM v1 graph capture
Unsloth: Could not patch vLLM V0 graph capture: No module named 'vllm.worker.model_runner'
==((====))==  Unsloth 2025.10.8: Fast Llama patching. Transformers: 4.55.4. vLLM: 0.11.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit with actual GPU utilization = 59.31%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 39.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 4096. Num Sequences = 288.
Unsloth: vLLM's KV Cache can use up to 22.36 GB. Also swap space = 6 GB.
WARN

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Unsloth 2025.10.8 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Model ready with LoRA.


### Dataset Loading and Prompt Formatting for GRPO

In this step, we load the unified dataset generated from the previous notebook (`final_dataset_finalissimo.json`)  
and convert it into a format compatible with GRPO-based training.

#### Process Overview
1. **Load the dataset** into a `DatasetDict` containing `train`, `validation`, and `test` splits.  
2. **Create structured prompts** following a *system + user* format to simulate conversational summarization instructions.  
   - The system role defines behavior and required output format.  
   - The user role specifies the summarization constraint (either a word or sentence count).  
3. **Dynamic instruction generation:**  
   - If the sample has a `split_word_X` label, the prompt requests a summary of exactly `X` words.  
   - If the sample has a `split_sentence_X` label, the prompt requests a summary of exactly `X` sentences.  
4. **Prompt length filtering:**  
   We remove exa\mples whose combined tokenized prompt exceeds the limit defined by  
   ```python
   MAX_PROMPT_LENGTH = MAX_SEQ_LENGTH - MAX_COMPLETION  # 4096 - 800 = 3296


In [8]:
with open("Dataset/summary_corpus_merged.json", "r", encoding="utf-8") as f:
    data = json.load(f)

dataset_dict = DatasetDict({
    'train': Dataset.from_list(data['train']),
    'validation': Dataset.from_list(data['val']),
    'test': Dataset.from_list(data['test'])
})
print(dataset_dict)

# Create user + system prompt structure
def format_summarization_prompt_dynamic(example):
    doc = example["document"]
    split_instruction = example["split"]

    example['prompt'] = []
    example['target_word_count'] = None
    example['target_sentence_count'] = None

    # ---- Add the system role first ----
    system_content = (
        "\nYou are a highly precise text processing assistant. Your primary function is to follow "
        "user instructions for summarization with perfect accuracy, especially regarding length constraints.\n\n"
        "You must always respond in the following XML format. Do not include any text outside of this tag.\n\n"
        "<summary>\n"
        "In this section, you will provide ONLY the final summary. The summary must strictly adhere "
        "to the user's constraints.\n"
        "</summary>\n"
    )
    example['prompt'].append({'role': 'system', 'content': system_content})

    # ---- Build the user role instruction ----
    if split_instruction.startswith('split_word_'):
        target_words = int(split_instruction.split('_')[-1])
        user_content = f"Summarize the following document in exactly {target_words} words.\n\nDocument:\n{doc}"
        example['prompt'].append({'role': 'user', 'content': user_content})
        example['target_word_count'] = target_words

    elif split_instruction.startswith('split_sentence_'):
        target_sentences = int(split_instruction.split('_')[-1])
        user_content = f"Summarize the following document in exactly {target_sentences} sentences.\n\nDocument:\n{doc}"
        example['prompt'].append({'role': 'user', 'content': user_content})
        example['target_sentence_count'] = target_sentences

    return example

def filter_by_full_prompt_length(dataset, tokenizer, max_prompt_len: int):
    def is_within_limit(example):
        # Concatenate messages into a single string
        full_prompt_text = "".join([msg["content"] for msg in example["prompt"]])
        return len(tokenizer(full_prompt_text, add_special_tokens=False)['input_ids']) <= max_prompt_len

    original_rows = len(dataset)
    filtered_dataset = dataset.filter(is_within_limit, num_proc=4)
    new_rows = len(filtered_dataset)

    print(f"Filtered by FULL prompt <= {max_prompt_len} tokens. "
          f"Original: {original_rows} → New: {new_rows} (Removed: {original_rows - new_rows})")
    return filtered_dataset


MAX_PROMPT_LENGTH = MAX_SEQ_LENGTH - MAX_COMPLETION  # 3296
formatted_dataset_dict = dataset_dict.map(format_summarization_prompt_dynamic)

formatted_dataset_dict = DatasetDict({
    split: filter_by_full_prompt_length(ds, tokenizer, MAX_PROMPT_LENGTH)
    for split, ds in formatted_dataset_dict.items()
})

# Save result
formatted_dataset_dict.save_to_disk("Dataset/processed_dataset_grpo")
print("✅ Dataset saved to 'processed_dataset_grpo'")
ds: DatasetDict = load_from_disk(DATASET_DISK_DIR)
print(ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'summary', 'document_word_count', 'summary_word_count', 'summary_sentence_count', 'split_word', 'split_sentence', 'dataset', 'split'],
        num_rows: 2571
    })
    validation: Dataset({
        features: ['id', 'document', 'summary', 'document_word_count', 'summary_word_count', 'summary_sentence_count', 'split_word', 'split_sentence', 'dataset', 'split'],
        num_rows: 300
    })
    test: Dataset({
        features: ['id', 'document', 'summary', 'document_word_count', 'summary_word_count', 'summary_sentence_count', 'split_word', 'split_sentence', 'dataset', 'split'],
        num_rows: 1040
    })
})


Map:   0%|          | 0/2571 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/1040 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/2571 [00:00<?, ? examples/s]

...

Filtered by FULL prompt <= 3296 tokens. Original: 1040 → New: 991 (Removed: 49)


Saving the dataset (0/1 shards):   0%|          | 0/2463 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/283 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/991 [00:00<?, ? examples/s]

✅ Dataset saved to 'processed_dataset_grpo'
DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'summary', 'document_word_count', 'summary_word_count', 'summary_sentence_count', 'split_word', 'split_sentence', 'dataset', 'split', 'prompt', 'target_word_count', 'target_sentence_count'],
        num_rows: 2463
    })
    validation: Dataset({
        features: ['id', 'document', 'summary', 'document_word_count', 'summary_word_count', 'summary_sentence_count', 'split_word', 'split_sentence', 'dataset', 'split', 'prompt', 'target_word_count', 'target_sentence_count'],
        num_rows: 283
    })
    test: Dataset({
        features: ['id', 'document', 'summary', 'document_word_count', 'summary_word_count', 'summary_sentence_count', 'split_word', 'split_sentence', 'dataset', 'split', 'prompt', 'target_word_count', 'target_sentence_count'],
        num_rows: 991
    })
})


### Reward Function Design for GRPO

GRPO relies on scalar reward signals to evaluate the quality of generated summaries.  
Here, we define multiple length and structure-based reward functions, designed to guide the model toward producing summaries that are:

1. **Structurally valid** — enclosed within `<summary>...</summary>` tags.  
2. **Length-consistent** — matching the requested word or sentence count.  

We implemented both penalty-based (negative) and reward-based (positive) formulations to explore different training dynamics:

| Combination | Description |
|--------------|--------------|
|  `reward_dynamic_length_negative` + `reward_summary_structure_negative` | Penalizes deviations and structure violations (strict, conservative). |
| `reward_length_squared_penalty` + `reward_summary_structure_negative` | Uses a quadratic penalty for large deviations — encourages precision. |
| `reward_dynamic_length_positive` + `reward_summary_structure_positive` | Rewards exact matches — encourages exploration with soft feedback. |

For GRPO training, we can easily select a reward setup by assigning it to a variable before initializing the trainer.


In [ ]:
def count_words(text):
    if not text:
        return 0
    return len(word_tokenize(text))

def count_sentences(text):
    if not text:
        return 0
    return len(sent_tokenize(text))

def extract_xml_answer(text: str) -> str:
    match = re.search(r"<summary>(.*?)</summary>", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

def reward_dynamic_length_negative(
    completions,
    target_word_count,
    target_sentence_count,
    tolerance=5,
    **kwargs
):
    """
    PENALTY-BASED reward for length deviations.
    If structure invalid (no <summary>...</summary>), abstain (0.0) to avoid double penalty.
    """
    scores = []
    for i, completion in enumerate(completions):
        target_w = target_word_count[i]
        target_s = target_sentence_count[i]
        response_text = completion[0]["content"]
        summary_text = extract_xml_answer(response_text)

        if summary_text is None:
            scores.append(0.0)
            continue

        if target_w is not None:
            num_words = count_words(summary_text)
            distance = abs(num_words - target_w)
            if distance <= tolerance:
                score = 0.0
            else:
                penalty = (distance - tolerance) / max(1, target_w)
                score = -min(1.0, penalty)
            scores.append(score)
            continue

        if target_s is not None:
            num_sents = count_sentences(summary_text)
            distance = abs(num_sents - target_s)
            if distance == 0:
                score = 0.0
            else:
                penalty = distance / max(1, target_s)
                score = -min(1.0, penalty)
            scores.append(score)
            continue

        scores.append(0.0)

    return scores

def reward_summary_structure_negative(completions, **kwargs):
    """
    Harsh, binary penalty if the whole output is NOT exactly <summary> ... </summary>.
    """
    scores = []
    pat = re.compile(r"^\s*<summary>.*?</summary>\s*$", re.DOTALL)
    for completion in completions:
        text = completion[0]["content"]
        if pat.search(text):
            scores.append(0.0)   # perfect (no penalty)
        else:
            scores.append(-1.0)  # structure failure
    return scores

def reward_length_squared_penalty(completions, target_word_count, target_sentence_count,
                                  word_tolerance=5, sentence_tolerance=0,
                                  word_scale=0.01, sentence_scale=0.1, **kwargs):
    """
    Quadratic penalty: distance squared. Heavily punishes large deviations.
    Different tolerances for words vs sentences since word counts naturally vary more.
    """
    scores = []
    for i, completion in enumerate(completions):
        target_w = target_word_count[i]
        target_s = target_sentence_count[i]
        response_text = completion[0]["content"]

        summary_text = extract_xml_answer(response_text)
        if summary_text is None:
            scores.append(0.0)
            continue

        if target_w is not None:
            num_words = count_words(summary_text)
            distance = abs(num_words - target_w)

            if distance <= word_tolerance:
                score = 0.0
            else:
                effective_distance = distance - word_tolerance
                penalty = (effective_distance ** 2) * word_scale
                score = -min(1.0, penalty)

            scores.append(score)
            continue

        if target_s is not None:
            num_sents = count_sentences(summary_text)
            distance = abs(num_sents - target_s)

            if distance <= sentence_tolerance:
                score = 0.0
            else:
                effective_distance = distance - sentence_tolerance
                penalty = (effective_distance ** 2) * sentence_scale
                score = -min(1.0, penalty)

            scores.append(score)
            continue

        scores.append(0.0)
    return scores

def reward_dynamic_length_positive(
    completions,
    target_word_count,
    target_sentence_count,
    tolerance=5,
    **kwargs):

    scores = []
    for i, completion in enumerate(completions):
        target_w = target_word_count[i]
        target_s = target_sentence_count[i]
        response_text = completion[0]["content"]
        summary_text = extract_xml_answer(response_text)

        if summary_text is None:
            scores.append(0.0)
            continue

        if target_w is not None:
            num_words = count_words(summary_text)
            distance = abs(num_words - target_w)
            if distance <= tolerance:
                score = 1.0  # perfect
            else:
                penalty = (distance - tolerance) / max(1, target_w)
                score = max(0.0, 1.0 - min(1.0, penalty))
            scores.append(score)
            continue

        if target_s is not None:
            num_sents = count_sentences(summary_text)
            distance = abs(num_sents - target_s)
            if distance == 0:
                score = 1.0
            else:
                penalty = distance / max(1, target_s)
                score = max(0.0, 1.0 - min(1.0, penalty))
            scores.append(score)
            continue

        scores.append(0.0)

    return scores


def reward_summary_structure_positive(completions, **kwargs):
    scores = []
    pat = re.compile(r"^\s*<summary>.*?</summary>\s*$", re.DOTALL)
    for completion in completions:
        text = completion[0]["content"]
        if pat.search(text):
            scores.append(1.0)  
        else:
            scores.append(0.0)  
    return scores

In [18]:
reward_combinations = {
    "strict_penalty": [
        reward_dynamic_length_negative,
        reward_summary_structure_negative,
    ],
    "quadratic_penalty": [
        reward_length_squared_penalty,
        reward_summary_structure_negative,
    ],
    "positive_reward": [
        reward_dynamic_length_positive,
        reward_summary_structure_positive,
    ],
}

selected_reward_setup = "strict_penalty"  # <-- change here to test others
selected_rewards = reward_combinations[selected_reward_setup]

print(f"Using reward setup: {selected_reward_setup}")


Using reward setup: strict_penalty


### GRPO Training Configuration and Execution

In this section, we define the training hyperparameters and configure the GRPO training loop.



In [ ]:
num_generation = 4
gradient_accumulation_steps = 4
learning_rate = 1e-5

run_name = f"grpo-penalty-rank-{LORA_RANK}-lr-{learning_rate}-gens-{num_generation}-acc-{gradient_accumulation_steps}-{RUN_NAME_SUFFIX}"
wandb.init(project=WANDB_PROJECT, name=run_name)


training_args = GRPOConfig(
    run_name=run_name,
    fp16=False,
    report_to="wandb",  
    output_dir=OUTPUT_DIR,
    use_vllm=True,
    vllm_gpu_memory_utilization=0.4,
    learning_rate=learning_rate,
    max_steps=1000,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=gradient_accumulation_steps,
    logging_steps=1,
    save_steps=100,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    max_grad_norm=0.1,
    num_generations=num_generation,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=reward_combinations[selected_reward_setup],
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
)

print("Starting training...")
trainer.train()
print("Training finished.")


# Save LoRA Adapters
model.save_lora(LORA_SAVE_DIR)
print(f"LoRA adapters saved to: {LORA_SAVE_DIR}")

### Model Evaluation and Metrics Logging

After GRPO training, we evaluate the fine-tuned model using the test split of the processed dataset.  
This stage measures how well the model generalizes to unseen examples and how closely it follows the summarization constraints.


- Evaluation is performed with LoRA adapters loaded, ensuring results reflect the fine-tuned parameters.  
- The sampling temperature is set to `0.1` for deterministic behavior during testing.  
- All metrics, including custom ones, are automatically pushed to the active W&B run summary.

The following cell runs the evaluation on the test dataset and prints aggregated metrics to confirm the model’s final performance.


In [ ]:
def evaluate_model(
    model,
    tokenizer,
    test_dataset: Dataset,
    wandb_project_name: str,
    wandb_run_name: str,
    lora_path: str = "grpo_saved_lora",
    batch_size: int = 8,
    word_tolerance: int = 5,
):
    """
    Evaluates the model + LoRA on a test dataset and logs metrics to WandB.
    Assumes dataset has: 'prompt', 'answer', 'target_word_count', 'target_sentence_count'.
    """
    print(f"Starting evaluation with run name: {wandb_run_name}")
    wandb.init(project=wandb_project_name, name=wandb_run_name, job_type="evaluation")

    sampling_params = SamplingParams(
        temperature=0.1,
        max_tokens=MAX_SEQ_LENGTH,  # upper bound for generations
    )

    scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

    total_examples = len(test_dataset)
    format_failures = 0
    valid_examples  = 0

    word_exact_matches     = 0
    word_within_tolerance  = 0
    word_distances         = []
    sent_exact_matches     = 0
    sent_distances         = []

    rouge1_scores = []
    rougeL_scores = []

    # Preload LoRA request (reuse same in-memory model)
    lora_request = model.load_lora(lora_path) if lora_path else None

    for i in tqdm(range(0, total_examples, batch_size), desc="Evaluating Batches"):
        batch = test_dataset[i:i + batch_size]
        chat_histories = batch["prompt"]

        prompts = [
            tokenizer.apply_chat_template(
                chat,
                tokenize=False,
                add_generation_prompt=True,
            ) for chat in chat_histories
        ]
        ground_truths = batch["summary"]
        target_words  = batch["target_word_count"]
        target_sents  = batch["target_sentence_count"]

        # Generate with fast path + LoRA
        outputs = model.fast_generate(
            prompts, sampling_params,
            lora_request=lora_request
        )

        # Per-sample metrics
        for j, output in enumerate(outputs):
            generated_text = output.outputs[0].text
            summary_text = extract_xml_answer(generated_text)

            if summary_text is None:
                format_failures += 1
                continue

            valid_examples += 1

            # ROUGE
            reference_summary = ground_truths[j]
            rouge_scores = scorer.score(reference_summary, summary_text)
            rouge1_scores.append(rouge_scores["rouge1"].fmeasure)
            rougeL_scores.append(rouge_scores["rougeL"].fmeasure)

            # Word length metrics
            tw = target_words[j]
            if tw is not None:
                n_words = count_words(summary_text)
                dist = abs(n_words - tw)
                word_distances.append(dist)
                if dist == 0:
                    word_exact_matches += 1
                if dist <= word_tolerance:
                    word_within_tolerance += 1

            # Sentence length metrics
            ts = target_sents[j]
            if ts is not None:
                n_sents = count_sentences(summary_text)
                dist = abs(n_sents - ts)
                sent_distances.append(dist)
                if dist == 0:
                    sent_exact_matches += 1

    # Aggregate metrics
    if valid_examples > 0:
        metrics = {
            "rouge1":              float(np.mean(rouge1_scores)),
            "rougeL":              float(np.mean(rougeL_scores)),
            "word_exact%":         word_exact_matches / valid_examples,
            "word_within_tol%":    word_within_tolerance / valid_examples,
            "avg_word_distance":   float(np.mean(word_distances) if word_distances else 0.0),
            "sent_exact%":         sent_exact_matches / valid_examples,
            "avg_sent_distance":   float(np.mean(sent_distances) if sent_distances else 0.0),
        }
    else:
        metrics = {
            "rouge1": 0.0, "rougeL": 0.0, "word_exact%": 0.0,
            "word_within_tol%": 0.0, "avg_word_distance": float("inf"),
            "sent_exact%": 0.0, "avg_sent_distance": float("inf"),
        }

    metrics.update({
        "format_failure_rate": format_failures / total_examples if total_examples else 1.0,
        "valid_examples":      valid_examples,
        "total_examples":      total_examples,
    })

    print("\n--- Evaluation Results ---")
    for k, v in metrics.items():
        print(f"{k:<22} -> {v:.4f}" if isinstance(v, float) else f"{k:<22} -> {v}")

    wandb.run.summary.update(metrics)
    wandb.finish()
    print("\nEvaluation finished and results logged to W&B.")
    return metrics


EVAL_RUN_NAME = f"eval-run-{run_name}"
test_dataset  = ds["test"]

metrics = evaluate_model(
    model=model,
    tokenizer=tokenizer,
    test_dataset=test_dataset,
    wandb_project_name=WANDB_PROJECT,
    wandb_run_name=EVAL_RUN_NAME,
    lora_path=LORA_SAVE_DIR,   # evaluate WITH LoRA as requested
    batch_size=16,
    word_tolerance=5,
)


### Per-Class Evaluation Analysis

To better understand model behavior across different summarization constraints,
we compute the same evaluation metrics for each unique target word count and target sentence count in the test set.

This analysis helps identify whether the model:
- Adheres more accurately to shorter or longer summaries.
- Maintains consistent quality (ROUGE) across varying length targets.
- Struggles with specific instruction types (e.g., sentence vs. word constraints).

In [44]:
def per_class_analysis(
    model,
    tokenizer,
    test_dataset: Dataset,
    lora_path: str = "grpo_saved_lora",
    batch_size: int = 8,
    word_tolerance: int = 5,
):
    print("\nStarting per-class analysis...")
    sampling_params = SamplingParams(
        temperature=0.1,
        max_tokens=MAX_SEQ_LENGTH,
    )
    scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    lora_request = model.load_lora(lora_path) if lora_path else None

    # Prepare containers
    class_metrics = defaultdict(lambda: {
        "count": 0,
        "rouge1": [],
        "rougeL": [],
        "word_exact": 0,
        "word_within_tol": 0,
        "avg_word_distance": [],
        "sent_exact": 0,
        "avg_sent_distance": [],
    })

    total_examples = len(test_dataset)

    for i in tqdm(range(0, total_examples, batch_size), desc="Evaluating per-class"):
        batch = test_dataset[i:i + batch_size]
        prompts = [
            tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
            for chat in batch["prompt"]
        ]

        outputs = model.fast_generate(prompts, sampling_params, lora_request=lora_request)
        ground_truths = batch["summary"]
        target_words = batch["target_word_count"]
        target_sents = batch["target_sentence_count"]

        for j, output in enumerate(outputs):
            gen_text = output.outputs[0].text
            summary_text = extract_xml_answer(gen_text)
            if summary_text is None:
                continue

            reference_summary = ground_truths[j]
            rouge = scorer.score(reference_summary, summary_text)

            # Determine class label (word or sentence target)
            class_label = None
            if target_words[j] is not None:
                class_label = f"word_{target_words[j]}"
            elif target_sents[j] is not None:
                class_label = f"sent_{target_sents[j]}"
            else:
                continue

            m = class_metrics[class_label]
            m["count"] += 1
            m["rouge1"].append(rouge["rouge1"].fmeasure)
            m["rougeL"].append(rouge["rougeL"].fmeasure)

            # Word-level analysis
            tw = target_words[j]
            if tw is not None:
                n_words = count_words(summary_text)
                dist = abs(n_words - tw)
                m["avg_word_distance"].append(dist)
                if dist == 0:
                    m["word_exact"] += 1
                if dist <= word_tolerance:
                    m["word_within_tol"] += 1

            # Sentence-level analysis
            ts = target_sents[j]
            if ts is not None:
                n_sents = count_sentences(summary_text)
                dist = abs(n_sents - ts)
                m["avg_sent_distance"].append(dist)
                if dist == 0:
                    m["sent_exact"] += 1

    # Aggregate and create dataframe
    records = []
    for cls, m in class_metrics.items():
        c = m["count"]
        if c == 0:
            continue
        records.append({
            "class": cls,
            "samples": c,
            "rouge1": np.mean(m["rouge1"]),
            "rougeL": np.mean(m["rougeL"]),
            "word_exact%": m["word_exact"] / c if c else 0,
            "word_within_tol%": m["word_within_tol"] / c if c else 0,
            "avg_word_distance": np.mean(m["avg_word_distance"]) if m["avg_word_distance"] else 0,
            "sent_exact%": m["sent_exact"] / c if c else 0,
            "avg_sent_distance": np.mean(m["avg_sent_distance"]) if m["avg_sent_distance"] else 0,
        })

    df = pd.DataFrame(records).sort_values("class")
    print("\n--- Per-Class Evaluation Results ---")
    display(df)

    df.to_csv("per_class_evaluation.csv", index=False)
    print("Saved per-class metrics to per_class_evaluation.csv")
    return df

# Run per-class analysis
df_class_metrics = per_class_analysis(
    model=model,
    tokenizer=tokenizer,
    test_dataset=test_dataset,
    lora_path=LORA_SAVE_DIR,
    batch_size=8,
)



Starting per-class analysis...


Evaluating per-class:   0%|          | 0/124 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Evaluating per-class:   1%|          | 1/124 [00:00<02:02,  1.00it/s]

...

Processed prompts:   0%|          | 0/7 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Evaluating per-class: 100%|██████████| 124/124 [09:18<00:00,  4.50s/it]


--- Per-Class Evaluation Results ---


,class,samples,rouge1,rougeL,word_exact%,word_within_tol%,avg_word_distance,sent_exact%,avg_sent_distance
9,sent_1,97,0.312160,0.230814,0.000000,0.000000,0.000000,1.000000,0.000000
7,sent_2,98,0.337724,0.219451,0.000000,0.000000,0.000000,0.989796,0.010204
6,sent_3,98,0.363746,0.220076,0.000000,0.000000,0.000000,0.836735,0.428571
8,sent_4,98,0.391354,0.238840,0.000000,0.000000,0.000000,0.928571,0.153061
10,sent_5,92,0.397495,0.229563,0.000000,0.000000,0.000000,0.826087,0.173913
11,sent_6,93,0.394780,0.222271,0.000000,0.000000,0.000000,0.634409,0.376344
1,word_100,91,0.390414,0.242036,0.032967,0.252747,19.835165,0.000000,0.000000
4,word_150,85,0.441019,0.247336,0.011765,0.129412,25.152941,0.000000,0.000000
2,word_200,58,0.459875,0.254742,0.034483,0.051724,28.189655,0.000000,0.000000
5,word_250,29,0.466175,0.251562,0.000000,0.034483,53.275862,0.000000,0.000000


Saved per-class metrics to per_class_evaluation.csv


### Sample Generation and Inspection

To qualitatively inspect the model’s behavior, we define a helper function that:

- Randomly samples one example from the test set, or optionally from a specific target class (e.g., `target_word_count = 150` or `target_sentence_count = 3`).  
- Generates a new summary using the GRPO-trained model with LoRA adapters.  
- The prompt, the model-generated summary, and the reference summary can then be inspected.

This allows manual verification of how well the model follows:
- The required XML structure (`<summary>...</summary>`)  
- The word or sentence length constraints used during training.

#### Available Target Values
You can refer to these values when sampling examples:

- **Target word counts:** `[50, 100, 150, 200, 250, 300]`  
- **Target sentence counts:** `[1, 2, 3, 4, 5, 6]`

In [29]:
unique_word_counts = sorted(set(x for x in ds["test"]["target_word_count"] if x is not None))
unique_sentence_counts = sorted(set(x for x in ds["test"]["target_sentence_count"] if x is not None))

print("Unique target_word_count values:")
print(unique_word_counts)

print("\nUnique target_sentence_count values:")
print(unique_sentence_counts)


Unique target_word_count values:
[50, 100, 150, 200, 250, 300]

Unique target_sentence_count values:
[1, 2, 3, 4, 5, 6]


In [ ]:
def sample_and_generate(
    model,
    tokenizer,
    dataset: Dataset,
    lora_path: str = "grpo_saved_lora",
    class_filter: str = None,
    temperature: float = 0.1,
):
    # Apply optional class filter
    if class_filter:
        subset = [ex for ex in dataset if class_filter in ex["split"]]
        if not subset:
            print(f"No samples found for class '{class_filter}'. Using random sample.")
            subset = dataset
    else:
        subset = dataset

    # Randomly select one example
    example = random.choice(subset)
    prompt_chat = example["prompt"]
    reference = example.get("summary", "")
    split_label = example.get("split", "unknown")

    formatted_prompt = tokenizer.apply_chat_template(
        prompt_chat,
        tokenize=False,
        add_generation_prompt=True
    )

    lora_request = model.load_lora(lora_path)

    sampling_params = SamplingParams(
        temperature=temperature,
        max_tokens=MAX_COMPLETION,
    )
    output = model.fast_generate([formatted_prompt], sampling_params, lora_request=lora_request)
    generated_text = output[0].outputs[0].text
    extracted_summary = extract_xml_answer(generated_text)

    return {
        "class": split_label,
        "prompt": formatted_prompt,
        "generated": extracted_summary,
        "reference": reference
    }

result = sample_and_generate(
    model=model,
    tokenizer=tokenizer,
    dataset=test_dataset,
    lora_path=LORA_SAVE_DIR,
    class_filter="3",   #class filter can be any number from the cell above
)


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [40]:
result['class']

'split_sentence_3'

In [41]:
result['prompt']

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 22 Oct 2025\n\nYou are a highly precise text processing assistant. Your primary function is to follow user instructions for summarization with perfect accuracy, especially regarding length constraints.\n\nYou must always respond in the following XML format. Do not include any text outside of this tag.\n\n<summary>\nIn this section, you will provide ONLY the final summary. The summary must strictly adhere to the user's constraints.\n</summary><|eot_id|><|start_header_id|>user<|end_header_id|>\n\nSummarize the following document in exactly 3 sentences.\n\nDocument:\nSECTION 1. SHORT TITLE.\n\n    This Act may be cited as the ``Judicial Transparency and Ethics \nEnhancement Act of 2006''.\n\nSEC. 2. INSPECTOR GENERAL FOR THE JUDICIAL BRANCH.\n\n    (a) Creation and Duties.--Part III of title 28, United States Code, \nis amended by adding at the end the following:\n\n        `

In [42]:
result['generated']

'The Inspector General for the Judicial Branch was established to oversee the judicial branch, conduct investigations, and prevent waste and abuse within the judicial system. The Inspector General will have the power to investigate and report on judicial misconduct, audit and supervise audits, and detect and prevent fraud and abuse. The Inspector General will also have the authority to obtain information and testimony from various government agencies and individuals, and to enforce laws and regulations through civil action.'

In [43]:
result['reference']

'Judicial Transparency and Ethics Enhancement Act of 2006 - Amends the federal judicial code to establish the Office of Inspector General for the Judicial Branch of the U.S. government, to be headed by an Inspector General appointed by the Chief Justice.\n\nRequires the Office to: (1) investigate matters pertaining to the Judicial Branch (other than the Supreme Court), including possible misconduct in office of justices and judges; (2) conduct and supervise audits and investigations; and (3) prevent and detect waste, fraud, and abuse.\n\nProvides for whistleblower protection.'